In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/rubabq66/invoice-receipt-contract-ocr/dataset_multi/metadata.json
/kaggle/input/datasets/rubabq66/invoice-receipt-contract-ocr/dataset_multi/boxes/RECEIPT_000022.box
/kaggle/input/datasets/rubabq66/invoice-receipt-contract-ocr/dataset_multi/boxes/CONTRACT_000061.box
/kaggle/input/datasets/rubabq66/invoice-receipt-contract-ocr/dataset_multi/boxes/RECEIPT_000099.box
/kaggle/input/datasets/rubabq66/invoice-receipt-contract-ocr/dataset_multi/boxes/INVOICE_000085.box
/kaggle/input/datasets/rubabq66/invoice-receipt-contract-ocr/dataset_multi/boxes/RECEIPT_000066.box
/kaggle/input/datasets/rubabq66/invoice-receipt-contract-ocr/dataset_multi/boxes/INVOICE_000000.box
/kaggle/input/datasets/rubabq66/invoice-receipt-contract-ocr/dataset_multi/boxes/RECEIPT_000040.box
/kaggle/input/datasets/rubabq66/invoice-receipt-contract-ocr/dataset_multi/boxes/RECEIPT_000063.box
/kaggle/input/datasets/rubabq66/invoice-receipt-contract-ocr/dataset_multi/boxes/INVOICE_000084.box
/kaggle/in

## *Week 8: Document Classification and API*

|*Name:*         |	Rubab Qaiser                                       |
|----------------|-----------------------------------------------------|
|*Course:*       |	Introduction to the Applied Artificial Intelligence|
|*Semester:*     |	BS 8th Semester                                    |
|*Week: *        |	Week 8                                             |
|*Project:*      |	Document Classification + Rest API Deployment      |
|*Lab Duration:* |	90 minutes                                         |

In [2]:
!pip install fastapi uvicorn python-multipart scikit-learn joblib

## *Task 1.1:DOCUMENT CLASSIFICATION*

In [3]:
import os
import pytesseract
from PIL import Image
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score
import joblib
def load_documents(data_dir):
    documents=[]
    labels=[]
    for doc_type in os.listdir(data_dir):
        folder_path=os.path.join(data_dir,doc_type)
        if not os.path.isdir(folder_path):
            continue
            
        for filename in os.listdir(folder_path):
            file_path=os.path.join(folder_path,filename)
            #extract text via ocr
            img=Image.open(file_path)
                
            text=pytesseract.image_to_string(img)
            documents.append(text)
            labels.append(doc_type)
    return documents,labels
            
    documents,labels=load_documents('training data')
    print(f'Loaded {len(documents)} documents')
    print(f'classes: {set(labels)}')
                

In [4]:
import os

print(os.getcwd())

/kaggle/working


In [5]:
os.listdir()

['__notebook__.ipynb']

In [6]:
for root, dirs, files in os.walk('.'):
    print("ROOT:", root)
    print("DIRS:", dirs)
    print("FILES:", files)
    print("-" * 40)

ROOT: .
DIRS: []
FILES: ['__notebook__.ipynb']
----------------------------------------


In [7]:
import os

os.listdir('/kaggle/input/datasets/rubabq66/invoice-receipt-contract-ocr')

['dataset_multi']

In [8]:
os.listdir('/kaggle/input/datasets/rubabq66/invoice-receipt-contract-ocr/dataset_multi')

['boxes', 'metadata.json', 'ground_truth', 'images']

In [9]:
import os
#os.listdir('/kaggle/input/')
for root,dirs,files in os.walk('/kaggle/input/'):
    print("ROOT:", root)
    print("DIRS:", dirs)
    print("FILES:", files[:5])
    print("-" * 40)

ROOT: /kaggle/input/
DIRS: ['datasets']
FILES: []
----------------------------------------
ROOT: /kaggle/input/datasets
DIRS: ['rubabq66']
FILES: []
----------------------------------------
ROOT: /kaggle/input/datasets/rubabq66
DIRS: ['invoice-receipt-contract-ocr']
FILES: []
----------------------------------------
ROOT: /kaggle/input/datasets/rubabq66/invoice-receipt-contract-ocr
DIRS: ['dataset_multi']
FILES: []
----------------------------------------
ROOT: /kaggle/input/datasets/rubabq66/invoice-receipt-contract-ocr/dataset_multi
DIRS: ['boxes', 'ground_truth', 'images']
FILES: ['metadata.json']
----------------------------------------
ROOT: /kaggle/input/datasets/rubabq66/invoice-receipt-contract-ocr/dataset_multi/boxes
DIRS: []
FILES: ['RECEIPT_000022.box', 'CONTRACT_000061.box', 'RECEIPT_000099.box', 'INVOICE_000085.box', 'RECEIPT_000066.box']
----------------------------------------
ROOT: /kaggle/input/datasets/rubabq66/invoice-receipt-contract-ocr/dataset_multi/ground_truth
D

In [10]:
import json

metadata_path = '/kaggle/input/datasets/rubabq66/invoice-receipt-contract-ocr/dataset_multi/metadata.json'

with open(metadata_path, 'r') as f:
    metadata = json.load(f)

print(type(metadata))
print(metadata[:2] if isinstance(metadata, list) else metadata)

<class 'list'>
[{'doc_id': 'RECEIPT_000000', 'image': 'images/RECEIPT_000000.png', 'box_file': 'boxes/RECEIPT_000000.box', 'ground_truth': 'ground_truth/RECEIPT_000000.txt', 'doc_type': 'receipt', 'store': 'Khaadi', 'subtotal': 29173.96, 'tax': 3500.88, 'total': 32674.84, 'date': '2026-04-01', 'time': '13:38:31'}, {'doc_id': 'RECEIPT_000001', 'image': 'images/RECEIPT_000001.png', 'box_file': 'boxes/RECEIPT_000001.box', 'ground_truth': 'ground_truth/RECEIPT_000001.txt', 'doc_type': 'receipt', 'store': 'Khaadi', 'subtotal': 18222.77, 'tax': 2186.73, 'total': 20409.5, 'date': '2026-03-23', 'time': '02:11:56'}]


In [11]:
import os
import json
import pytesseract

from PIL import Image

In [12]:
def load_documents(base_path):

    documents = []
    labels = []

    metadata_path = os.path.join(base_path, 'metadata.json')

    with open(metadata_path, 'r') as f:
        metadata = json.load(f)

    for item in metadata:

        image_path = os.path.join(base_path, item['image'])

        label = item['doc_type']

        try:
            img = Image.open(image_path)

            text = pytesseract.image_to_string(img)

            documents.append(text)

            labels.append(label)

        except Exception as e:
            print(f"Error processing {image_path}: {e}")

    return documents, labels

In [13]:
base_path = '/kaggle/input/datasets/rubabq66/invoice-receipt-contract-ocr/dataset_multi'

documents, labels = load_documents(base_path)

print(f'Loaded {len(documents)} documents')

print(f'Classes: {set(labels)}')

Loaded 300 documents
Classes: {'invoice', 'contract', 'receipt'}


In [14]:
print(load_documents)

<function load_documents at 0x7d596f7ffce0>


### *Task 1.2: Train Classifier*

In [15]:
X_train,X_test,y_train,y_test=train_test_split(documents,labels,test_size=0.2,random_state=42,stratify=labels)
#create TF-IDF vectorizer
vectorizer=TfidfVectorizer(max_features=1000, stop_words='english',
                          ngram_range=(1,2)) #unigram,bigram
#fit and transform
X_train_vec=vectorizer.fit_transform(X_train)
#train classifier
X_test_vec=vectorizer.transform(X_test)
classifier=LogisticRegression(max_iter=1000)
classifier.fit(X_train_vec,y_train)
#evaluate
y_pred=classifier.predict(X_test_vec)
accuracy=accuracy_score(y_test,y_pred)
print(f'Accuracy: {accuracy:.2%}')
print('\nClassification Report:')
print(classification_report(y_test,y_pred))

Accuracy: 100.00%

Classification Report:
              precision    recall  f1-score   support

    contract       1.00      1.00      1.00        20
     invoice       1.00      1.00      1.00        20
     receipt       1.00      1.00      1.00        20

    accuracy                           1.00        60
   macro avg       1.00      1.00      1.00        60
weighted avg       1.00      1.00      1.00        60



### *Task 1.3: Save Model*

In [16]:
import os
import joblib

# Create models directory
os.makedirs('models', exist_ok=True)

# Save files
joblib.dump(vectorizer, 'models/vectorizer.pkl')
joblib.dump(classifier, 'models/classifier.pkl')

print('Model Saved Successfully')

Model Saved Successfully


In [17]:
os.makedirs('models', exist_ok=True)

In [18]:
import joblib

# Load vectorizer
vectorizer = joblib.load('/kaggle/working/models/vectorizer.pkl')

# Load classifier
classifier = joblib.load('/kaggle/working/models/classifier.pkl')

print("Models Loaded Successfully")

Models Loaded Successfully
